# Using OGC Dimensions: FAO ASIS Example

The **Agricultural Stress Index System (ASIS)** monitors vegetation health globally using MODIS/VIIRS satellite imagery at dekadal (10-day) resolution since 1984.

ASIS is a multi-dimensional datacube with:
- **time** — dekadal temporal dimension (36 periods/year, 2,700+ members since 1950)
- **season** — growing season (GS1, GS2)
- **land_cover** — land cover type (Cropland, Grassland)
- **x, y** — spatial axes (global coverage)

This notebook demonstrates how to:
1. Configure a STAC collection with the mixed pagination pattern
2. Browse paginated temporal members
3. Use inverse lookups and search
4. Compare with the legacy gismgr API

## 1. The Mixed Pattern

Not all dimensions need pagination. ASIS has:

| Dimension | Type | Cardinality | Strategy |
|-----------|------|------------|----------|
| `time` | temporal | ~2,736 (76 years x 36) | `size` + `href` + `generator` |
| `season` | nominal | 2 | inline `values` |
| `land_cover` | nominal | 2 | inline `values` |
| `x`, `y` | spatial | continuous | `extent` + `reference_system` |

**Rule of thumb:** dimensions with < ~1,000 members use `values` arrays; larger ones use pagination via `href`.

## 2. STAC Collection Configuration

The STAC plugin config for the ASI-D collection is set via the Config API. The `cube_dimensions` map defines all five dimensions with their appropriate strategy.

In [ ]:
import json

# This is the payload for: PUT /configs/catalogs/asis/collections/ASI-D/stac
stac_config = {
    "enabled": True,
    "enabled_extensions": ["datacube"],
    "cube_dimensions": {
        "time": {
            "type": "temporal",
            "description": (
                "Dekadal temporal dimension. Each month splits into "
                "D1 (days 1-10), D2 (days 11-20), D3 (days 21-end). "
                "36 periods per year."
            ),
            "extent": ["1950-01-01T00:00:00Z", "2026-03-31T00:00:00Z"],
            "step": None,
            "unit": "dekad",
            "size": 2736,
            "href": "/dimensions/temporal-dekadal/members",
            "generator": {
                "type": "daily-period",
                "config": {"period_days": 10, "scheme": "monthly"},
                "invertible": True,
                "capabilities": ["generate", "extent", "inverse", "search"],
                "search_protocols": ["exact", "range"],
            },
        },
        "season": {
            "type": "nominal",
            "description": "Growing season",
            "values": ["GS1", "GS2"],
        },
        "land_cover": {
            "type": "nominal",
            "description": "Land cover type",
            "values": ["LC-C", "LC-G"],
        },
        "x": {
            "type": "spatial",
            "axis": "x",
            "extent": [-180.0, 179.996],
            "reference_system": 4326,
        },
        "y": {
            "type": "spatial",
            "axis": "y",
            "extent": [-56.004, 75.004],
            "reference_system": 4326,
        },
    },
}

print(json.dumps(stac_config, indent=2))

In [ ]:
# To apply this configuration (uncomment to run against a live instance):

# import httpx
#
# BASE_URL = "https://data.review.fao.org/geospatial/search"
#
# response = httpx.put(
#     f"{BASE_URL}/configs/catalogs/asis/collections/ASI-D/stac",
#     json=stac_config,
#     headers={"Authorization": "Bearer <your-token>"},
# )
# print(response.status_code, response.json())

## 3. Resulting STAC Collection

After configuration, `GET /stac/catalogs/asis/collections/ASI-D` returns a STAC Collection with `cube:dimensions`:

In [ ]:
# Expected cube:dimensions in the STAC collection response
expected_cube_dimensions = {
    "time": {
        "type": "temporal",
        "description": "Dekadal temporal dimension. Each month splits into D1 (days 1-10), D2 (days 11-20), D3 (days 21-end). 36 periods per year.",
        "extent": ["1950-01-01T00:00:00Z", "2026-03-31T00:00:00Z"],
        "unit": "dekad",
        "size": 2736,
        "href": "/dimensions/temporal-dekadal/members",
        "generator": {
            "type": "daily-period",
            "config": {"period_days": 10, "scheme": "monthly"},
            "invertible": True,
            "capabilities": ["generate", "extent", "inverse", "search"],
            "search_protocols": ["exact", "range"],
        },
    },
    "season": {
        "type": "nominal",
        "description": "Growing season",
        "values": ["GS1", "GS2"],
    },
    "land_cover": {
        "type": "nominal",
        "description": "Land cover type",
        "values": ["LC-C", "LC-G"],
    },
    "x": {
        "type": "spatial",
        "axis": "x",
        "extent": [-180.0, 179.996],
        "reference_system": 4326,
    },
    "y": {
        "type": "spatial",
        "axis": "y",
        "extent": [-56.004, 75.004],
        "reference_system": 4326,
    },
}

print("cube:dimensions in STAC collection response:")
print(json.dumps(expected_cube_dimensions, indent=2))

## 4. Browsing Dekadal Members (Pagination)

The `time` dimension has `size: 2736` and an `href` pointing to the paginated members endpoint.

Clients follow OGC API pagination conventions: `limit` + `offset` query params, `rel:next` / `rel:prev` links in the response.

In [ ]:
# Local demonstration using the generator directly
from ogc_dimensions.generators import DailyPeriodGenerator

dekadal = DailyPeriodGenerator(period_days=10, scheme="monthly")

# Page 1: first 5 members from 1950
page1 = dekadal.generate("1950-01-01", "2026-03-31", limit=5, offset=0)
print(f"Page 1 (offset=0, limit=5):")
print(f"  numberMatched: {page1.number_matched}")
print(f"  numberReturned: {len(page1.members)}")
for m in page1.members:
    print(f"    {m.code}  {m.start} -> {m.end}")

# Page 2: next 5 members
page2 = dekadal.generate("1950-01-01", "2026-03-31", limit=5, offset=5)
print(f"\nPage 2 (offset=5, limit=5):")
for m in page2.members:
    print(f"    {m.code}  {m.start} -> {m.end}")

In [ ]:
# Via HTTP (uncomment to run against a live instance):

# import httpx
#
# BASE_URL = "https://data.review.fao.org/geospatial/search"
#
# # First page
# r = httpx.get(f"{BASE_URL}/dimensions/temporal-dekadal/members", params={"limit": 5, "offset": 0})
# data = r.json()
# print(f"numberMatched: {data['numberMatched']}")
# print(f"numberReturned: {data['numberReturned']}")
# for feat in data["features"]:
#     print(f"  {feat['id']}  {feat['properties']['dimension:start']} -> {feat['properties']['dimension:end']}")
#
# # Follow rel:next link
# next_link = next(l for l in data["links"] if l["rel"] == "next")
# r2 = httpx.get(next_link["href"])
# print(f"\nNext page: {[f['id'] for f in r2.json()['features']]}")

## 5. Inverse Lookup

Given an observation date, find the dekad it belongs to. This is key for joining satellite imagery (daily) with dekadal aggregates.

In [ ]:
# Local: which dekad contains a given date?
test_dates = ["2024-01-15", "2024-02-28", "2024-06-21", "2024-12-31"]

for date in test_dates:
    inv = dekadal.inverse(date)
    print(f"  {date} -> {inv.member}  (range: {inv.range})")

In [ ]:
# Via HTTP (uncomment to run against a live instance):

# r = httpx.get(f"{BASE_URL}/dimensions/temporal-dekadal/inverse", params={"value": "2024-01-15"})
# print(r.json())
# # Expected: {"member": "2024-K02", "range": ["2024-01-11", "2024-01-20"]}

## 6. Temporal Range Search

Filter dekadal members by date range — returns only the dekads that overlap with the query interval.

In [ ]:
# Local: dekads in H1 2024
h1_2024 = dekadal.generate("2024-01-01", "2024-06-30", limit=100)

print(f"Dekads in H1 2024: {h1_2024.number_matched}")
print(f"Expected: 18 (6 months x 3 dekads/month)")
print()
for m in h1_2024.members[:6]:  # show first two months
    print(f"  {m.code}  {m.start} -> {m.end}")
print(f"  ... and {h1_2024.number_matched - 6} more")

In [ ]:
# Via HTTP (uncomment to run against a live instance):

# r = httpx.get(
#     f"{BASE_URL}/dimensions/temporal-dekadal/members",
#     params={"datetime": "2024-01-01/2024-06-30", "limit": 100},
# )
# data = r.json()
# print(f"numberMatched: {data['numberMatched']}")
# for feat in data["features"][:6]:
#     print(f"  {feat['id']}")

## 7. Small Dimensions (Inline Values)

Season and land cover have only 2 members each — no pagination needed. Clients read `values` directly from `cube:dimensions`.

In [ ]:
# These are read directly from the STAC collection response — no API call needed
season_values = expected_cube_dimensions["season"]["values"]
lc_values = expected_cube_dimensions["land_cover"]["values"]

print(f"Season dimension:     {season_values}  (read from cube:dimensions.season.values)")
print(f"Land cover dimension: {lc_values}  (read from cube:dimensions.land_cover.values)")
print()
print("No pagination needed — clients enumerate these directly.")
print(f"Time dimension:       size={expected_cube_dimensions['time']['size']} -> follow href for pagination")

## 8. Pentadal Variant (CHIRPS)

The same pattern applies to pentadal (5-day) products like CHIRPS rainfall estimates. Only the generator config changes: `period_days: 5` instead of `10`, giving 72 periods per year.

In [ ]:
pentadal = DailyPeriodGenerator(period_days=5, scheme="monthly")

# Compare period counts
dek_year = dekadal.generate("2024-01-01", "2024-12-31", limit=100)
pen_year = pentadal.generate("2024-01-01", "2024-12-31", limit=100)

print(f"Dekadal (ASIS):  {dek_year.number_matched} periods/year")
print(f"Pentadal (CHIRPS): {pen_year.number_matched} periods/year")
print()

# CHIRPS cube_dimensions.time would look like:
chirps_time_dim = {
    "type": "temporal",
    "description": "Pentadal temporal dimension (5-day periods, 72/year)",
    "extent": ["1981-01-01T00:00:00Z", "2026-03-31T00:00:00Z"],
    "unit": "pentad",
    "size": pen_year.number_matched * 45,  # ~45 years
    "href": "/dimensions/temporal-pentadal/members",
    "generator": {
        "type": "daily-period",
        "config": {"period_days": 5, "scheme": "monthly"},
        "invertible": True,
        "capabilities": ["generate", "extent", "inverse", "search"],
        "search_protocols": ["exact", "range"],
    },
}
print(f"CHIRPS time dimension config:")
print(json.dumps(chirps_time_dim, indent=2))

## 9. Legacy vs OGC: Format Comparison

The old gismgr API at `data.apps.fao.org` serves dimension members in a proprietary format. The new OGC Dimensions API uses standard GeoJSON FeatureCollection.

In [ ]:
# Legacy gismgr format (proprietary)
legacy_response = {
    "code": "POS-DEKAD",
    "caption": "Dekad",
    "type": "WHAT",
    "hasNext": True,
    "members": [
        {"code": "D01", "caption": "Dekad 01 (Jan 01 - Jan 10)"},
        {"code": "D02", "caption": "Dekad 02 (Jan 11 - Jan 20)"},
        {"code": "D03", "caption": "Dekad 03 (Jan 21 - Jan 31)"},
    ],
}

# OGC Dimensions format (standard GeoJSON FeatureCollection)
ogc_response = {
    "type": "FeatureCollection",
    "numberMatched": 2736,
    "numberReturned": 3,
    "features": [
        {
            "type": "Feature",
            "id": "1950-K01",
            "geometry": None,
            "properties": {
                "dimension:start": "1950-01-01",
                "dimension:end": "1950-01-10",
                "time": {"interval": ["1950-01-01", "1950-01-10"]},
            },
        },
        {
            "type": "Feature",
            "id": "1950-K02",
            "geometry": None,
            "properties": {
                "dimension:start": "1950-01-11",
                "dimension:end": "1950-01-20",
                "time": {"interval": ["1950-01-11", "1950-01-20"]},
            },
        },
        {
            "type": "Feature",
            "id": "1950-K03",
            "geometry": None,
            "properties": {
                "dimension:start": "1950-01-21",
                "dimension:end": "1950-01-31",
                "time": {"interval": ["1950-01-21", "1950-01-31"]},
            },
        },
    ],
    "links": [
        {"rel": "self", "href": "/dimensions/temporal-dekadal/members?limit=3&offset=0"},
        {"rel": "next", "href": "/dimensions/temporal-dekadal/members?limit=3&offset=3"},
    ],
}

print("=== Legacy gismgr format ===")
print(json.dumps(legacy_response, indent=2))
print()
print("=== OGC Dimensions format ===")
print(json.dumps(ogc_response, indent=2))

Key differences:

| Aspect | Legacy (gismgr) | OGC Dimensions |
|--------|----------------|----------------|
| Format | Proprietary JSON | GeoJSON FeatureCollection |
| Pagination | `hasNext` boolean | `numberMatched` + `rel:next` links |
| Member IDs | `D01`-`D36` (year-agnostic) | `2024-K01` (year-specific) |
| Temporal bounds | Caption text only | `dimension:start` / `dimension:end` |
| Type system | `"WHAT"` for all | `temporal`, `nominal`, `spatial` |
| Discoverable | No | Yes (generator in `cube:dimensions`) |

## Summary: When to Paginate

```
Dimension cardinality?
  |
  +-- < ~1,000 members --> use "values" array (inline)
  |     Examples: season (2), land cover (2), admin level 0 (5)
  |
  +-- >= ~1,000 members --> use "size" + "href" + "generator"
        Examples: dekadal since 1950 (2,736), pentadal since 1981 (3,240),
                  country admin level 2 (3,500+)
```

The `generator` metadata makes the pagination machine-discoverable: clients know the algorithm type, whether inverse lookup is supported, and which search protocols are available — without downloading all members.